In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

data_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # تكبير الصور لتناسب ResNet
    transforms.RandomHorizontalFlip(), 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=data_transforms)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, pin_memory=True)

def build_model(num_classes):
    # تحميل ResNet18 بـ Weights جاهزة
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    # تجميد الطبقات (Freeze) عشان نستخدمها كمستخرج ميزات فقط
    for param in model.parameters():
        param.requires_grad = False
    
    # استبدال الطبقة الأخيرة (Head) لتناسب عدد فئات مشروعك
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    return model

model = build_model(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
# تحسين أوزان الطبقة الأخيرة فقط
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
def train_model(model, train_loader, criterion, optimizer, epochs=2):
    model.train()
    print("Starting Fine-tuning on GPU...")
    
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # Backward pass & Optimize
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            if i % 50 == 49:
                print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(train_loader)}], Loss: {running_loss/50:.4f}")
                running_loss = 0.0

    print("Finished Fine-tuning!")

# تنفيذ التدريب
train_model(model, train_loader, criterion, optimizer)

Using device: cpu
Starting Fine-tuning on GPU...
Epoch [1/2], Step [50/782], Loss: 1.4405
Epoch [1/2], Step [100/782], Loss: 0.8765
Epoch [1/2], Step [150/782], Loss: 0.8062
Epoch [1/2], Step [200/782], Loss: 0.7902
Epoch [1/2], Step [250/782], Loss: 0.7528
Epoch [1/2], Step [300/782], Loss: 0.7611
Epoch [1/2], Step [350/782], Loss: 0.7281
Epoch [1/2], Step [400/782], Loss: 0.7457
Epoch [1/2], Step [450/782], Loss: 0.7178
Epoch [1/2], Step [500/782], Loss: 0.6861
Epoch [1/2], Step [550/782], Loss: 0.6922
Epoch [1/2], Step [600/782], Loss: 0.6584
Epoch [1/2], Step [650/782], Loss: 0.6663
Epoch [1/2], Step [700/782], Loss: 0.6600
Epoch [1/2], Step [750/782], Loss: 0.6605
Epoch [2/2], Step [50/782], Loss: 0.6564
Epoch [2/2], Step [100/782], Loss: 0.6730
Epoch [2/2], Step [150/782], Loss: 0.6257
Epoch [2/2], Step [200/782], Loss: 0.6489
Epoch [2/2], Step [250/782], Loss: 0.6905
Epoch [2/2], Step [300/782], Loss: 0.6687
Epoch [2/2], Step [350/782], Loss: 0.6417
Epoch [2/2], Step [400/782], 